In [1]:
# Init
import pandas as pd
import numpy as np
import os

OUTPUT_ROOT = "Benchmark_data"

splits = [0.5, 0.6, 0.8, 1.0]

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

# Label mapping for multi-class
LABEL_MAP = {
    "normal":    0,
    "dodag":     1,
    "flooding":  2,
    "blackhole": 3,
    "rank":      4,
    # Also handle if already numeric (as string)
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
}


def temporal_subset(df, time_col, ratio):
    df = df.sort_values(time_col).reset_index(drop=True)
    n = int(len(df) * ratio)
    return df.iloc[:n]


def split_train_val_test(df):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))

    train = df.iloc[:train_end]
    val   = df.iloc[train_end:val_end]
    test  = df.iloc[val_end:]

    return train, val, test


def convert_multiclass(df):
    """Map string labels to numeric 0-4 for multi-class."""
    df = df.copy()
    df["label"] = df["label"].apply(
        lambda x: LABEL_MAP.get(str(x).lower(), -1)  # -1 if unknown
    )
    unknown = (df["label"] == -1).sum()
    if unknown > 0:
        print(f"  WARNING: {unknown} unknown labels found in multi-class conversion!")
    return df


def convert_binary(df):
    """Map labels to 0 (Normal) or 1 (Attack)."""
    df = df.copy()
    df["label"] = df["label"].apply(
        lambda x: 0 if str(x).lower() in ["normal", "0"] else 1
    )
    return df


def save_split(dataset_name, split_ratio, train, val, test, mode):
    folder = f"{OUTPUT_ROOT}/{dataset_name}/{mode}/{int(split_ratio * 100)}"
    os.makedirs(folder, exist_ok=True)

    train.to_csv(f"{folder}/train.csv", index=False)
    val.to_csv(f"{folder}/val.csv",   index=False)
    test.to_csv(f"{folder}/test.csv",  index=False)

# Data split for data 1

In [2]:
dataset_name = "dataset1_new"
path = "Raw_data/1"

files = [
    "blackhole.csv",
    "dodag.csv",
    "flooding.csv",
    "rank.csv"
]

dfs = []
for f in files:
    df = pd.read_csv(f"{path}/{f}")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print("Total samples:", len(df))

for r in splits:
    subset = temporal_subset(df, "time", r)
    train, val, test = split_train_val_test(subset)

    # ✅ Multi-class: convert string → numeric (0-4)
    train_m = convert_multiclass(train)
    val_m   = convert_multiclass(val)
    test_m  = convert_multiclass(test)
    save_split(dataset_name, r, train_m, val_m, test_m, "multi")

    # ✅ Binary: convert string → 0 or 1
    train_b = convert_binary(train)
    val_b   = convert_binary(val)
    test_b  = convert_binary(test)
    save_split(dataset_name, r, train_b, val_b, test_b, "binary")

    print(f"Finished {int(r*100)}%")

Total samples: 1639975
Finished 50%
Finished 60%
Finished 80%
Finished 100%


# Data split for data 3

In [3]:
dataset_name = "dataset3_new"
path = "Raw_data/3"

files = ["RPL-IDS-Beh.csv"]

dfs = []
for f in files:
    df = pd.read_csv(f"{path}/{f}")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print("Total samples:", len(df))

for r in splits:
    subset = temporal_subset(df, "time_sec", r)
    train, val, test = split_train_val_test(subset)

    # ✅ Multi-class
    train_m = convert_multiclass(train)
    val_m   = convert_multiclass(val)
    test_m  = convert_multiclass(test)
    save_split(dataset_name, r, train_m, val_m, test_m, "multi")

    # ✅ Binary
    train_b = convert_binary(train)
    val_b   = convert_binary(val)
    test_b  = convert_binary(test)
    save_split(dataset_name, r, train_b, val_b, test_b, "binary")

    print(f"Finished {int(r*100)}%")

Total samples: 158254
Finished 50%
Finished 60%
Finished 80%
Finished 100%


# Data split for data 4